# PCA component -> decoded position analysis

**Direction:** `research/directions/pca-component-position-analysis.md` (`[in-frame]`, sub-Q 1+3).

**Question.** The PCA-component waterfall explorer suggested PC0 +/-3 sigma visibly shifts *one*
object while leaving the other in place. The waterfall shows intensities, not positions.
Do PCA components map onto **decoded object positions**, and does any single component
selectively move one object and not the other?

This notebook is **self-contained**: it reconstructs `states_tf`, `subspace`, `warm`, `linear`,
`decode_pos`, `rollout_from_flat`, `sigma` from scratch (it does NOT rely on a live
`editability_structure.ipynb` kernel). The **printed tables** are the load-bearing deliverable
(per the brief, this is table-centric so results are verifiable without viewing figures).

---
## 1 - Setup: model, probe, state-manifold PCA, warm-up to edit

In [ ]:
import sys
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import pim.eval as eval
from pim.extractors import LinearExtractor, StateDefinition, identity_mse, hungarian_mse
from pim.editors import fit_state_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda"
BATCH_SIZE      = 512
NUM_WORKERS     = 6

N_OBJ         = 2
USE_HUNGARIAN = False      # fixed reflectivities -> identity matching
SUBSPACE_VAR  = 0.90

# Sweep config
N_SWEEP = 64                                       # warmed-up base states swept
N_ROLL  = 10                                       # rollout length (>= step 10 persistence)
PCS     = list(range(6))                           # PC 0..5
ALPHAS  = np.array([-3,-2,-1,0,1,2,3], dtype=float)

In [ ]:
model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# Teacher-force the whole test set -> bank of visited hidden states.
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)

# Linear position probe.
state_def = StateDefinition(name="positions", state_shape=(N_OBJ, 2),
                            extract_fn=lambda b: b["positions"])
env_states_tf = test.positions[:, :-1, :N_OBJ, :]
vis_mask_tf   = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
loss_fn       = hungarian_mse if USE_HUNGARIAN else identity_mse

linear = LinearExtractor(model.hidden_size, state_def, use_lstsq=True)
train_mse = linear.fit(states_tf, env_states_tf, mask=vis_mask_tf, loss_fn=loss_fn, device=DEVICE)
linear = linear.to(DEVICE).eval()

# Global state-manifold PCA subspace + per-direction data-std (sigma).
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
Xc = torch.from_numpy(states_tf.reshape(-1, model.hidden_size)).float()
Xc = Xc - Xc.mean(0)
def sigma(d):                       # data-std along a unit direction
    return float((Xc @ d).std())

print(f"Model  : {ckpt_info.run_name}  (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"states_tf={states_tf.shape}  probe train MSE={train_mse:.6f}")
print(f"subspace: kept {subspace.n_components}/{subspace.hidden_size} comps "
      f"({subspace.total_explained:.4f} var)")

In [ ]:
# Warm up to the edit frame -> base hidden states we will perturb along PCs.
N = min(N_SWEEP, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame,
                            n_viz=1, n_ctx_show=8, device=DEVICE)
h_base = warm.h_at_edit[:N]                          # (N, H)

@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    """Roll out from each flat state; step 0 = decode (no advance)."""
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout)
        obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

@torch.no_grad()
def decode_pos(h_array):
    t = torch.as_tensor(h_array, dtype=torch.float32, device=DEVICE)
    return linear(t).cpu().numpy()                  # (..., N_OBJ, 2)

print(f"h_base={h_base.shape}  edit_frame={edits.edit_frame}")

---
## 2 - Core sweep: PC x alpha -> decoded positions

For each PC i=0..5 and alpha in {-3..3}: add `alpha * sigma_i * PC_i` to every base state,
roll out `N_ROLL` steps, decode positions. `mean_pos[pc]` is the sample-mean decoded
position, shape `(n_alpha, N_ROLL, N_OBJ, 2)`.

In [ ]:
sweep, sigmas = {}, {}
for pc_i in PCS:
    pc = subspace.basis[:, pc_i].detach().cpu()
    s_pc = sigma(pc); sigmas[pc_i] = s_pc
    pc_np = pc.numpy()
    per_alpha = []
    for a in ALPHAS:
        h_edit = h_base + (a * s_pc) * pc_np
        _, hs = rollout_from_flat(h_edit, N_ROLL)
        per_alpha.append(decode_pos(hs))            # (N, N_ROLL, N_OBJ, 2)
    sweep[pc_i] = np.stack(per_alpha)               # (n_alpha, N, N_ROLL, N_OBJ, 2)
    print(f"PC{pc_i}: sigma={s_pc:.4f}  swept {len(ALPHAS)} alphas")

mean_pos = {pc_i: sweep[pc_i].mean(axis=1) for pc_i in PCS}   # (n_alpha, N_ROLL, N_OBJ, 2)

---
## 3 - Sensitivity table  (slope d decoded_pos / d alpha)

Rows = PC index, cols = obj0-x, obj0-y, obj1-x, obj1-y. `np.polyfit(alphas, pos, 1)[0]`.
`|d obj|` is the per-object displacement-vector slope magnitude; `ratio = max/min`.
A PC is flagged **selective** when ratio >= 3 *and* the larger object actually moves
(|d| > 0.02). Units: world-units of decoded position per 1 sigma along the PC.

In [ ]:
COORD = ["obj0-x", "obj0-y", "obj1-x", "obj1-y"]

def slope_table(step):
    tab = np.zeros((len(PCS), 4))
    for r, pc_i in enumerate(PCS):
        flat = mean_pos[pc_i][:, step].reshape(len(ALPHAS), 4)
        for c in range(4):
            tab[r, c] = np.polyfit(ALPHAS, flat[:, c], 1)[0]
    return tab

def print_table(tab, title):
    print(title)
    print(f"{'PC':>3} " + " ".join(f"{c:>9}" for c in COORD) +
          f" | {'|d obj0|':>9} {'|d obj1|':>9} {'ratio':>7} {'selective?':>10}")
    for r, pc_i in enumerate(PCS):
        o0 = np.hypot(tab[r,0], tab[r,1]); o1 = np.hypot(tab[r,2], tab[r,3])
        big, small = max(o0,o1), min(o0,o1)
        ratio = big/small if small > 1e-9 else np.inf
        sel = "YES" if ratio >= 3.0 and big > 0.02 else "no"
        print(f"{pc_i:>3} " + " ".join(f"{tab[r,c]:>9.4f}" for c in range(4)) +
              f" | {o0:>9.4f} {o1:>9.4f} {ratio:>7.2f} {sel:>10}")

tab0 = slope_table(0)
print_table(tab0, "SENSITIVITY TABLE - slope at STEP 0 (rows=PC, cols=decoded-pos slope per obj-coord)")

---
## 4 - 2D scatter: decoded positions across the alpha sweep (PC0, PC1)

Coloured by object. A straight, separated line for one object only would be a selective
edit. The per-coordinate R^2 of position-vs-alpha quantifies linearity.

In [ ]:
for pc_i in [0, 1]:
    mp = mean_pos[pc_i][:, 0]                        # (n_alpha, N_OBJ, 2) at step 0
    print(f"\nPC{pc_i}:  alpha ->  obj0(x,y)   obj1(x,y)")
    for ai, a in enumerate(ALPHAS):
        print(f"  a={a:+.0f}: obj0=({mp[ai,0,0]:+.3f},{mp[ai,0,1]:+.3f})  "
              f"obj1=({mp[ai,1,0]:+.3f},{mp[ai,1,1]:+.3f})")
    for oi in range(2):
        for ci, cn in enumerate(["x", "y"]):
            y = mp[:, oi, ci]; p = np.polyfit(ALPHAS, y, 1); yhat = np.polyval(p, ALPHAS)
            ss_res = ((y-yhat)**2).sum(); ss_tot = ((y-y.mean())**2).sum()
            r2 = 1 - ss_res/ss_tot if ss_tot > 1e-12 else 1.0
            print(f"    obj{oi}-{cn}: slope={p[0]:+.4f}  R^2={r2:.3f}")

# Scatter figure (PC0, PC1): decoded positions across the alpha sweep, coloured by object.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, pc_i in zip(axes, [0, 1]):
    mp = mean_pos[pc_i][:, 0]
    sc = ax.scatter(mp[:,0,0], mp[:,0,1], c=ALPHAS, cmap="coolwarm", marker="o",
                    s=60, edgecolor="k", label="obj0")
    ax.scatter(mp[:,1,0], mp[:,1,1], c=ALPHAS, cmap="coolwarm", marker="^",
               s=70, edgecolor="k", label="obj1")
    ax.plot(mp[:,0,0], mp[:,0,1], "0.5", lw=0.8); ax.plot(mp[:,1,0], mp[:,1,1], "0.5", lw=0.8)
    ax.set_title(f"PC{pc_i}: decoded pos vs alpha (o=obj0, ^=obj1; color=alpha)")
    ax.set_xlabel("decoded x"); ax.set_ylabel("decoded y"); ax.grid(alpha=0.3); ax.legend()
fig.colorbar(sc, ax=axes, label="alpha (sigma)")
display(fig); plt.close(fig)

---
## 5 - Persistence: slope at step 0 vs 5 vs last

Does the per-PC displacement hold or revert over the rollout? (Same reversion question
applied to PCA directions.) Note: step-0 selectivity is the direct-edit signature; later
steps mix in the learned dynamics.

In [ ]:
steps_check = [0, 5, min(9, N_ROLL-1)]
tabs = {s: slope_table(s) for s in steps_check}

print(f"{'PC':>3} | " + "   ".join(f"s{s}:obj0 obj1" for s in steps_check))
for r, pc_i in enumerate(PCS):
    parts = []
    for s in steps_check:
        t = tabs[s]
        o0 = np.hypot(t[r,0], t[r,1]); o1 = np.hypot(t[r,2], t[r,3])
        parts.append(f"{o0:5.3f} {o1:5.3f}")
    print(f"{pc_i:>3} | " + "   ".join(parts))

print()
print_table(tabs[steps_check[-1]], f"slopes at last step {steps_check[-1]} (persistence check)")

---
## 6 - Bonus: render decoded positions vs the model's own rollout (PC0)

For PC0, alpha in {-3,0,3}: feed decoded positions through `pim.simulator.renderer` and
compare the *rendered-from-decoded* waterfall against the model's autoregressive rollout obs.
Low RMS => decoded positions are geometrically consistent with what the model generates.

**Important control:** the alpha=0 (no-edit) row is the floor. If it is already large, the
mismatch is a probe-vs-physical-position gap, not edit-induced. Reflectivities are the fixed
`[refl_min, refl_max]` (edits split carries no reflectivities; fixed_reflectivities=True makes
them deterministic). obs_noise_std=0 so we compare to the model's denoised prediction.

In [ ]:
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene

sim = test.config["dataset"]["sim"]
def make_cfg(n_frames):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"],
                     n_frames=n_frames, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                     fixed_reflectivities=True, obs_noise_std=0.0, boundary="open",
                     always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)   # obj0=min, obj1=max
RAD  = np.array([sim["radius"]] * N_OBJ, dtype=np.float32)
COL  = np.tile(np.array([[1,1,1]], dtype=np.float32), (N_OBJ, 1))

def rms(a, b): return float(np.sqrt(((a - b) ** 2).mean()))

pc = subspace.basis[:, 0].detach().cpu(); s_pc = sigma(pc); pc_np = pc.numpy()
cfg = make_cfg(N_ROLL)
print("BONUS: RMS(rendered-from-decoded  vs  model AR rollout)   PC0")
for a in [-3.0, 0.0, 3.0]:
    obs_model, hs = rollout_from_flat(h_base + (a*s_pc)*pc_np, N_ROLL)
    pos = decode_pos(hs)
    rendered = np.zeros_like(obs_model)
    for i in range(N):
        vel = np.zeros((N_ROLL, N_OBJ, 2), dtype=np.float32)
        scene = Scene(positions=pos[i].astype(np.float32), velocities=vel,
                      radii=RAD, colors=COL, reflectivities=REFL, config=cfg)
        _, _, inten = render_scene(scene)
        rendered[i] = inten
    per_step = np.sqrt(((rendered - obs_model) ** 2).mean(axis=(0, 2)))
    print(f"alpha={a:+.0f}:  RMS={rms(rendered, obs_model):.4f}   per-step: "
          + " ".join(f"{v:.3f}" for v in per_step))

obs0, _ = rollout_from_flat(h_base, N_ROLL)
perm = np.random.RandomState(1).permutation(N)
print(f"\nreference scale (shuffle): RMS(model vs other-sample model rollout) = {rms(obs0, obs0[perm]):.4f}")

---
## 7 - Verdict

See `research/scratch/2026-06-23-pca-component-position.md` for the written observation and
the promotion flag. One-line: at the **direct-edit step (step 0)** no single PC is selective
(all object-displacement ratios < 2.5, PC0/PC1 move both objects together in x) -> the
high-variance PCA directions encode global scene shifts, consistent with the
probe-sigma << PCA-sigma gap. Object-selective slopes only emerge *later* in the rollout
(PC2, PC4 at the last step) as a dynamical effect, not a clean editable direction.

# =====================================================================
## 8 - OBSERVATION-SPACE EXTENSION (sub-Q2/Q3): does PC0 move the dim object only?

**Why.** Sections 1-7 worked entirely in *decoded-position* space and concluded PC0 moves
**both** objects' x together (a global shift). But the human's read of the intensity
**waterfalls** disagrees: along PC0, only the **dim** object (obj0 = `refl_min`, identity
fixed via `USE_HUNGARIAN=False`) appears to move while the **bright** object (obj1 = `refl_max`)
stays put. Decoded-position space and observation space *disagree*.

This section surfaces that disagreement **in observation space** (the space the model actually
generates), without trying to resolve it:

- **8a** overlays the model's generated 1D intensity scans across the PC0 alpha sweep, so the
  dim feature (lower intensity) and bright feature (higher intensity) are individually trackable.
- **8b** attributes the per-alpha *observation change* to dim vs bright object, two ways:
  (i) **intensity-band** assignment (model-only, no renderer), (ii) **obs_id** assignment from a
  renderer reference. Printed as a TABLE and plotted.
- **8c** puts decoded-position change per object next to observation-change per object for PC0
  (the explicit reconciliation).
- **8d** keeps/extends the PC0 -3sigma / unsteered / +3sigma waterfall panels.

> NOTE: this extension does **not** answer whether the bright object's decoded motion is real
> or a probe artifact. It only makes the disagreement legible. The open question is flagged for
> Sevan in `research/scratch/`.

In [ ]:
# ---- 8 setup: PC0 sweep keeping the MODEL-GENERATED observations + a renderer reference ----
import os
os.makedirs("/tmp/pca_ext", exist_ok=True)

PC0           = subspace.basis[:, 0].detach().cpu()
S_PC0         = sigma(PC0)
PC0_np        = PC0.numpy()
ALPHAS_EXT    = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)
A0_IDX        = int(np.where(ALPHAS_EXT == 0)[0][0])   # unsteered index
OBS_RES       = test.obs.shape[-1]

# Reflectivity bands (obj0 = dim = refl_min, obj1 = bright = refl_max).
REFL_DIM, REFL_BRIGHT = float(sim["refl_min"]), float(sim["refl_max"])
print(f"OBS_RES={OBS_RES}  dim(obj0) refl={REFL_DIM:.3f}  bright(obj1) refl={REFL_BRIGHT:.3f}")

# For each alpha along PC0: model-generated obs (N, N_ROLL, R), decoded pos (N, N_ROLL, N_OBJ, 2),
# AND a renderer reference (obs_id, obs_intensity) rendered FROM the decoded positions.
sweep_obs   = np.zeros((len(ALPHAS_EXT), N, N_ROLL, OBS_RES), dtype=np.float32)  # model output
sweep_pos   = np.zeros((len(ALPHAS_EXT), N, N_ROLL, N_OBJ, 2), dtype=np.float32) # decoded pos
sweep_rid   = np.zeros((len(ALPHAS_EXT), N, N_ROLL, OBS_RES), dtype=np.int64)    # renderer obs_id
sweep_rint  = np.zeros((len(ALPHAS_EXT), N, N_ROLL, OBS_RES), dtype=np.float32)  # renderer intensity

cfg_ext = make_cfg(N_ROLL)   # reuse the bonus-section renderer config (sec 6)
for ai, a in enumerate(ALPHAS_EXT):
    h_edit = h_base + (a * S_PC0) * PC0_np
    obs_m, hs = rollout_from_flat(h_edit, N_ROLL)
    pos = decode_pos(hs)
    sweep_obs[ai] = obs_m
    sweep_pos[ai] = pos
    for i in range(N):
        vel = np.zeros((N_ROLL, N_OBJ, 2), dtype=np.float32)
        scene = Scene(positions=pos[i].astype(np.float32), velocities=vel,
                      radii=RAD, colors=COL, reflectivities=REFL, config=cfg_ext)
        _, rid, rint = render_scene(scene)
        sweep_rid[ai, i]  = rid
        sweep_rint[ai, i] = rint
    print(f"alpha={a:+.0f}: model-obs + decoded-pos + renderer-ref done")

print("sweep_obs", sweep_obs.shape, " sweep_pos", sweep_pos.shape, " sweep_rid", sweep_rid.shape)

### 8a - 1D intensity-scan overlays across the PC0 alpha sweep

For a few representative samples, at one or two fixed rollout frames, overlay the model's
**generated** 1D intensity scan (intensity vs ray index) for alpha in {-3..+3}sigma along PC0.
The bright object is a high-intensity feature (~`refl_max`), the dim object a lower-intensity
feature (~`refl_min`). Watch which feature *slides* with alpha (color = alpha) and which stays.

Samples chosen so that at the unsteered frame the model shows two clearly separated features
(one dim, one bright) — i.e. both objects visible and resolvable.

In [ ]:
# Pick representative samples: at the unsteered frame, both objects visible & well-separated.
# Use the renderer reference at alpha=0 to find samples where both obj ids appear with a gap.
FRAMES_SHOW = [0, 5]                 # fixed rollout frames to overlay
def both_visible_and_separated(i, frame):
    rid = sweep_rid[A0_IDX, i, frame]
    has0 = (rid == 0).any(); has1 = (rid == 1).any()
    if not (has0 and has1):
        return False, 0.0
    c0 = np.where(rid == 0)[0].mean(); c1 = np.where(rid == 1)[0].mean()
    return True, abs(c0 - c1)

cand = []
for i in range(N):
    ok, gap = both_visible_and_separated(i, FRAMES_SHOW[0])
    if ok and gap >= 6:              # require a visible spatial gap between the two features
        cand.append((gap, i))
cand.sort(reverse=True)
SAMPLES_SHOW = [i for _, i in cand[:3]] if cand else [0, 1, 2]
print("representative samples:", SAMPLES_SHOW)

rays = np.arange(OBS_RES)
cmap = plt.cm.coolwarm
norm = plt.Normalize(ALPHAS_EXT.min(), ALPHAS_EXT.max())

fig, axes = plt.subplots(len(SAMPLES_SHOW), len(FRAMES_SHOW),
                         figsize=(6.2 * len(FRAMES_SHOW), 3.0 * len(SAMPLES_SHOW)),
                         squeeze=False)
for r, smp in enumerate(SAMPLES_SHOW):
    for c, fr in enumerate(FRAMES_SHOW):
        ax = axes[r][c]
        for ai, a in enumerate(ALPHAS_EXT):
            ax.plot(rays, sweep_obs[ai, smp, fr], color=cmap(norm(a)),
                    lw=2.0 if a == 0 else 1.2, alpha=1.0 if a == 0 else 0.8,
                    zorder=3 if a == 0 else 2)
        # Mark renderer-derived object centroids at unsteered (where the two features sit at a=0).
        rid0 = sweep_rid[A0_IDX, smp, fr]
        for oid, lab, ls in [(0, "dim(obj0)", ":"), (1, "bright(obj1)", "--")]:
            if (rid0 == oid).any():
                cx = np.where(rid0 == oid)[0].mean()
                ax.axvline(cx, color="k", ls=ls, lw=1.0, alpha=0.6)
                ax.text(cx, 1.02, lab, rotation=90, va="bottom", ha="center", fontsize=7)
        ax.axhline(REFL_DIM, color="0.4", ls=":", lw=0.7)
        ax.axhline(REFL_BRIGHT, color="0.4", ls=":", lw=0.7)
        ax.set_title(f"sample {smp}, frame {fr}")
        ax.set_xlabel("ray index"); ax.set_ylabel("model intensity")
        ax.set_ylim(-0.02, 1.08); ax.grid(alpha=0.25)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
fig.colorbar(sm, ax=axes, label="alpha (sigma along PC0)", shrink=0.8)
fig.suptitle("PC0 alpha-sweep: model-generated 1D intensity scans (dashed = unsteered obj centroids)",
             y=1.005, fontsize=12)
fig.savefig("/tmp/pca_ext/8a_intensity_scans.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/pca_ext/8a_intensity_scans.png")

### 8b - Per-object observation-change attribution

How much of the model's observation change under PC0 is due to the **dim** vs the **bright**
object? Each ray of the model's generated scan is attributed to one object, then we report the
**RMS change** of the model intensity (vs the unsteered alpha=0 scan) *within each object's rays*,
as a function of alpha. Two independent ray-assignment methods (they should roughly agree):

- **(i) intensity-band** (model-only, no renderer): assign each ray to dim/bright by which
  reference reflectivity its *unsteered* intensity is closer to (rays near 0 = background/miss
  are dropped). Robust to where the renderer thinks objects are.
- **(ii) obs_id** (renderer reference): assign each ray using the renderer's `obs_id` computed
  from the **unsteered** decoded positions (so the ray->object map is fixed at alpha=0 and we
  measure how the model's intensity on *those same rays* changes with alpha).

A large dim-object RMS with a small bright-object RMS would confirm "dim moves, bright stays"
in observation space. Reported as RMS over (samples x frames x rays-in-band). Printed + plotted.

In [ ]:
# Build per-ray object-assignment masks at the UNSTEERED alpha (fixed map), then measure
# the model's per-band RMS intensity change vs unsteered across alpha.
obs0 = sweep_obs[A0_IDX]                                  # (N, N_ROLL, R) unsteered model obs
INTEN_FLOOR = 0.05                                       # below this = background/miss, drop

# (i) intensity-band masks (model-only): nearest reference reflectivity at unsteered.
d_dim    = np.abs(obs0 - REFL_DIM)
d_bright = np.abs(obs0 - REFL_BRIGHT)
mask_band_dim    = (obs0 > INTEN_FLOOR) & (d_dim <= d_bright)     # (N, N_ROLL, R)
mask_band_bright = (obs0 > INTEN_FLOOR) & (d_bright <  d_dim)

# (ii) obs_id masks (renderer reference) at unsteered decoded positions.
rid0 = sweep_rid[A0_IDX]                                  # (N, N_ROLL, R)
mask_id_dim    = (rid0 == 0)
mask_id_bright = (rid0 == 1)

def band_rms_vs_alpha(mask):
    """RMS of model-intensity change (vs unsteered) within `mask`, per alpha."""
    out = np.zeros(len(ALPHAS_EXT))
    m = mask.astype(bool)
    denom = m.sum()
    if denom == 0:
        return out
    for ai in range(len(ALPHAS_EXT)):
        diff = (sweep_obs[ai] - obs0)[m]
        out[ai] = np.sqrt((diff ** 2).mean())
    return out

rms_band_dim    = band_rms_vs_alpha(mask_band_dim)
rms_band_bright = band_rms_vs_alpha(mask_band_bright)
rms_id_dim      = band_rms_vs_alpha(mask_id_dim)
rms_id_bright   = band_rms_vs_alpha(mask_id_bright)

# Also a whole-scan RMS for context.
rms_all = np.array([np.sqrt(((sweep_obs[ai] - obs0) ** 2).mean()) for ai in range(len(ALPHAS_EXT))])

def print_attr(title, dim, bright):
    print(title)
    print(f"  rays assigned: dim={int(_last_denoms[0])}  bright={int(_last_denoms[1])}")
    print(f"  {'alpha':>6} {'dim RMS':>9} {'bright RMS':>11} {'dim/bright':>11}")
    for ai, a in enumerate(ALPHAS_EXT):
        ratio = dim[ai] / bright[ai] if bright[ai] > 1e-9 else np.inf
        print(f"  {a:>+6.0f} {dim[ai]:>9.4f} {bright[ai]:>11.4f} {ratio:>11.2f}")
    # summary: mean over |alpha|>=2 (the strong-edit regime)
    strong = np.abs(ALPHAS_EXT) >= 2
    md, mb = dim[strong].mean(), bright[strong].mean()
    print(f"  [|alpha|>=2 mean]  dim={md:.4f}  bright={mb:.4f}  dim/bright={md/mb if mb>1e-9 else np.inf:.2f}")
    print()

_last_denoms = (mask_band_dim.sum(), mask_band_bright.sum())
print_attr("METHOD (i) intensity-band attribution  [model-only]", rms_band_dim, rms_band_bright)
_last_denoms = (mask_id_dim.sum(), mask_id_bright.sum())
print_attr("METHOD (ii) obs_id attribution  [renderer reference]", rms_id_dim, rms_id_bright)
print(f"whole-scan RMS vs unsteered, per alpha: " + " ".join(f"{v:.3f}" for v in rms_all))

In [ ]:
# Plot per-object observation-change attribution (both methods), + a centroid-shift cross-check.
# Centroid cross-check: track the ray-index centroid of each renderer-id band's MODEL intensity
# (intensity-weighted) vs alpha — a direct "does the feature slide?" readout in obs space.
def band_centroid_vs_alpha(mask_oid):
    """Intensity-weighted ray centroid of the model scan within renderer-id band, per alpha,
    averaged over (samples, frames) where the band exists at unsteered."""
    out = np.full(len(ALPHAS_EXT), np.nan)
    m0 = mask_oid                                      # (N,N_ROLL,R) fixed unsteered map
    valid = m0.any(axis=-1)                            # (N,N_ROLL) band present
    for ai in range(len(ALPHAS_EXT)):
        cents = []
        oi = sweep_obs[ai]
        for n in range(N):
            for f in range(N_ROLL):
                if not valid[n, f]:
                    continue
                w = oi[n, f] * m0[n, f]
                if w.sum() <= 1e-6:
                    continue
                cents.append((rays * w).sum() / w.sum())
        if cents:
            out[ai] = np.mean(cents)
    return out

cen_dim    = band_centroid_vs_alpha(mask_id_dim)
cen_bright = band_centroid_vs_alpha(mask_id_bright)
# Centroid SHIFT relative to unsteered (rays per sigma readout):
shift_dim    = cen_dim    - cen_dim[A0_IDX]
shift_bright = cen_bright - cen_bright[A0_IDX]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
# Panel 1: intensity-band RMS
ax = axes[0]
ax.plot(ALPHAS_EXT, rms_band_dim,    "o-", color="#0072B2", label="dim (obj0)")
ax.plot(ALPHAS_EXT, rms_band_bright, "s-", color="#D55E00", label="bright (obj1)")
ax.set_title("(i) intensity-band:\nmodel obs RMS change per object"); ax.set_xlabel("alpha (sigma)")
ax.set_ylabel("RMS intensity change vs unsteered"); ax.grid(alpha=0.3); ax.legend()
# Panel 2: obs_id RMS
ax = axes[1]
ax.plot(ALPHAS_EXT, rms_id_dim,    "o-", color="#0072B2", label="dim (obj0)")
ax.plot(ALPHAS_EXT, rms_id_bright, "s-", color="#D55E00", label="bright (obj1)")
ax.set_title("(ii) obs_id (renderer ref):\nmodel obs RMS change per object"); ax.set_xlabel("alpha (sigma)")
ax.set_ylabel("RMS intensity change vs unsteered"); ax.grid(alpha=0.3); ax.legend()
# Panel 3: centroid slide cross-check
ax = axes[2]
ax.plot(ALPHAS_EXT, shift_dim,    "o-", color="#0072B2", label="dim (obj0)")
ax.plot(ALPHAS_EXT, shift_bright, "s-", color="#D55E00", label="bright (obj1)")
ax.axhline(0, color="0.6", lw=0.8)
ax.set_title("feature centroid SLIDE (cross-check):\nintensity-weighted ray centroid shift")
ax.set_xlabel("alpha (sigma)"); ax.set_ylabel("ray-index shift vs unsteered"); ax.grid(alpha=0.3); ax.legend()
fig.suptitle("PC0: per-object OBSERVATION-space change (dim vs bright)", y=1.02, fontsize=13)
fig.savefig("/tmp/pca_ext/8b_attribution.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

# slope of centroid slide (rays per sigma) — concise numeric readout
sl_dim    = np.polyfit(ALPHAS_EXT[~np.isnan(shift_dim)],    shift_dim[~np.isnan(shift_dim)], 1)[0]
sl_bright = np.polyfit(ALPHAS_EXT[~np.isnan(shift_bright)], shift_bright[~np.isnan(shift_bright)], 1)[0]
print(f"feature centroid slide (rays per sigma):  dim={sl_dim:+.3f}   bright={sl_bright:+.3f}   "
      f"|dim|/|bright|={abs(sl_dim)/abs(sl_bright) if abs(sl_bright)>1e-9 else np.inf:.2f}")
print("saved /tmp/pca_ext/8b_attribution.png")

### 8c - Reconciliation: decoded-position change vs observation change, per object (PC0)

The explicit comparison. For each object, side by side:

- **decoded-position change** = `||decoded_pos(alpha) - decoded_pos(0)||` (mean over samples,
  step 0), i.e. how far the *probe* says the object moved.
- **observation change** = the obs_id-band RMS intensity change from 8b, i.e. how much the
  *model's generated scan* changed on that object's rays.

If decoded space says "both move" while observation space says "only dim moves", the two
columns disagree for the bright object — that is the disagreement we are surfacing (not resolving).

In [ ]:
# Decoded-position displacement per object vs unsteered (step 0), mean over samples.
pos0 = sweep_pos[A0_IDX, :, 0]                            # (N, N_OBJ, 2) unsteered decoded pos
dpos = np.zeros((len(ALPHAS_EXT), N_OBJ))
for ai in range(len(ALPHAS_EXT)):
    d = sweep_pos[ai, :, 0] - pos0                        # (N, N_OBJ, 2)
    dpos[ai] = np.sqrt((d ** 2).sum(-1)).mean(0)          # mean ||disp|| per object

# Slopes (per sigma) for a one-number-per-object summary.
def slope(y): return np.polyfit(ALPHAS_EXT, y, 1)[0]
dpos_slope_dim    = slope(dpos[:, 0])                     # not signed-meaningful (it's a norm); use |disp| at a=+3 too
dpos_slope_bright = slope(dpos[:, 1])

# Signed x-slope from the existing sec-3 table (decoded x is the dominant axis for PC0).
sx_dim    = np.polyfit(ALPHAS_EXT, sweep_pos[:, :, 0, 0, 0].mean(1), 1)[0]   # obj0-x slope
sx_bright = np.polyfit(ALPHAS_EXT, sweep_pos[:, :, 0, 1, 0].mean(1), 1)[0]   # obj1-x slope

print("=== PC0 RECONCILIATION: decoded-position change  vs  observation change (per object) ===\n")
print(f"{'':>14} | {'DIM (obj0)':>22} | {'BRIGHT (obj1)':>22}")
print(f"{'alpha':>14} | {'dpos_norm':>10} {'obs RMS':>11} | {'dpos_norm':>10} {'obs RMS':>11}")
for ai, a in enumerate(ALPHAS_EXT):
    print(f"{a:>+14.0f} | {dpos[ai,0]:>10.4f} {rms_id_dim[ai]:>11.4f} | "
          f"{dpos[ai,1]:>10.4f} {rms_id_bright[ai]:>11.4f}")
print()
print(f"decoded x-slope (world-units/sigma):   dim={sx_dim:+.4f}   bright={sx_bright:+.4f}   "
      f"(ratio |dim|/|bright|={abs(sx_dim)/abs(sx_bright) if abs(sx_bright)>1e-9 else np.inf:.2f})")
strong = np.abs(ALPHAS_EXT) >= 2
print(f"obs_id RMS [|alpha|>=2 mean]:           dim={rms_id_dim[strong].mean():.4f}   "
      f"bright={rms_id_bright[strong].mean():.4f}   "
      f"(ratio dim/bright={rms_id_dim[strong].mean()/rms_id_bright[strong].mean():.2f})")
print(f"intensity-band RMS [|alpha|>=2 mean]:   dim={rms_band_dim[strong].mean():.4f}   "
      f"bright={rms_band_bright[strong].mean():.4f}   "
      f"(ratio dim/bright={rms_band_dim[strong].mean()/rms_band_bright[strong].mean():.2f})")

# Side-by-side bar reconciliation at alpha=+3 (and -3 for symmetry).
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, ai, a in zip(axes, [len(ALPHAS_EXT)-1, 0], [ALPHAS_EXT[-1], ALPHAS_EXT[0]]):
    x = np.arange(2)
    w = 0.35
    # normalize each metric to its own max across objects so they're comparable on one axis
    dp = dpos[ai] / (dpos[ai].max() + 1e-9)
    ob = np.array([rms_id_dim[ai], rms_id_bright[ai]])
    ob = ob / (ob.max() + 1e-9)
    ax.bar(x - w/2, dp, w, label="decoded-pos change (norm.)", color="#009E73")
    ax.bar(x + w/2, ob, w, label="obs change (obs_id, norm.)", color="#CC79A7")
    ax.set_xticks(x); ax.set_xticklabels(["dim (obj0)", "bright (obj1)"])
    ax.set_title(f"PC0 alpha={a:+.0f}: decoded-pos vs obs change\n(each normalized to its own max)")
    ax.set_ylabel("relative change"); ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
fig.suptitle("PC0 reconciliation: does the bright object move in obs space the way the probe says?",
             y=1.03, fontsize=12)
fig.savefig("/tmp/pca_ext/8c_reconciliation.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("\nsaved /tmp/pca_ext/8c_reconciliation.png")

### 8d - PC0 waterfall panels (-3sigma / unsteered / +3sigma)

The model-generated intensity waterfalls (rollout frame on y, ray index on x, grayscale =
intensity, "what the model sees") for the same representative samples, at PC0 alpha in
{-3, 0, +3}sigma. This is the human's original read surface. Renderer-derived object centroids
at unsteered are overlaid (dotted = dim, dashed = bright) so you can see which streak moves.

In [ ]:
# Waterfall panels: rows = representative samples, cols = alpha in {-3, 0, +3}.
ALPHA_PANEL = [-3.0, 0.0, 3.0]
panel_idx = [int(np.where(ALPHAS_EXT == a)[0][0]) for a in ALPHA_PANEL]

fig, axes = plt.subplots(len(SAMPLES_SHOW), len(ALPHA_PANEL),
                         figsize=(3.6 * len(ALPHA_PANEL), 3.0 * len(SAMPLES_SHOW)),
                         squeeze=False)
for r, smp in enumerate(SAMPLES_SHOW):
    for c, (ai, a) in enumerate(zip(panel_idx, ALPHA_PANEL)):
        ax = axes[r][c]
        wf = sweep_obs[ai, smp]                          # (N_ROLL, R)
        ax.imshow(wf, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                  interpolation="nearest")
        # Unsteered renderer centroids per frame for each object (overlay).
        rid_smp = sweep_rid[A0_IDX, smp]                 # (N_ROLL, R)
        for oid, color, ls in [(0, "#00BFFF", ":"), (1, "#FFA500", "--")]:
            xs, ys = [], []
            for f in range(N_ROLL):
                m = rid_smp[f] == oid
                if m.any():
                    xs.append(np.where(m)[0].mean()); ys.append(f)
            if xs:
                ax.plot(xs, ys, color=color, ls=ls, lw=1.3, alpha=0.9)
        ax.set_title(f"smp {smp}, alpha={a:+.0f}", fontsize=9)
        ax.set_xlabel("ray"); ax.set_ylabel("frame")
axes[0][0].plot([], [], color="#00BFFF", ls=":", label="dim(obj0) centroid @a=0")
axes[0][0].plot([], [], color="#FFA500", ls="--", label="bright(obj1) centroid @a=0")
axes[0][0].legend(loc="upper right", fontsize=7)
fig.suptitle("PC0 waterfalls (model-generated intensity): -3sigma / unsteered / +3sigma",
             y=1.01, fontsize=12)
fig.savefig("/tmp/pca_ext/8d_waterfalls.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/pca_ext/8d_waterfalls.png")